# OOD Main3 Consistency Ablation: Cross-Run Paper Tables

In [ ]:
from __future__ import annotations

from typing import Any

import pandas as pd

try:
    from IPython import get_ipython
    from IPython.display import Markdown, display
except ImportError:  # pragma: no cover
    get_ipython = None
    Markdown = None

    def display(obj: Any) -> None:
        print(obj)


import ood_main3_consistency_precomputed_tables_lib as ood_tables


pd.options.display.max_columns = 200


def md(text: str) -> None:
    shell_name = ""
    if get_ipython is not None and get_ipython() is not None:
        shell_name = get_ipython().__class__.__name__
    if Markdown is not None and shell_name == "ZMQInteractiveShell":
        display(Markdown(text))
    else:
        print(text)


inventory_df, summary_df, metrics_df = ood_tables.load_cross_run_bundle_frames()


## Cross-Run Paper Tables

Standard errors are shown when the underlying rows are available.

- If multiple bundles exist for the same model / feature set, the SE is computed across bundle means.
- Otherwise the SE falls back to the underlying source-environment validation rows or OOD transfer rows.
- In the environment tables, `Validation AUROC` is averaged over the source-validation rows that feed transfer into the listed held-out environment.


In [ ]:
if inventory_df.empty:
    md(
        "_No populated bundle directories were found. "
        "Set `OOD_MAIN3_PRECOMPUTED_BUNDLE_ROOTS` to one or more saved output directories if needed._"
    )
else:
    for target_name, target_title in ood_tables.target_rows(summary_df, metrics_df):
        md(f"### {target_title}")

        core_table = ood_tables.build_feature_summary_table(
            summary_df,
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.CORE_FEATURE_ORDER,
        )
        md("#### 1. Core Feature Summary")
        display(ood_tables.style_metric_table(core_table))
        core_missing_note = ood_tables.render_missing_feature_note(ood_tables.missing_feature_sets(core_table))
        if core_missing_note is not None:
            md(core_missing_note)

        core_env_table = ood_tables.build_environment_summary_table(
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.CORE_FEATURE_ORDER,
        )
        md("#### 2. Core Feature Summary By Held-Out Environment")
        display(ood_tables.style_metric_table(core_env_table))

        attention_subset_table = ood_tables.build_feature_summary_table(
            summary_df,
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.ATTENTION_SUBSET_FEATURE_ORDER,
        )
        md("#### 3. Attention-Only Subset Summary")
        display(ood_tables.style_metric_table(attention_subset_table))
        attention_missing_note = ood_tables.render_missing_feature_note(ood_tables.missing_feature_sets(attention_subset_table))
        if attention_missing_note is not None:
            md(attention_missing_note)

        attention_env_table = ood_tables.build_environment_summary_table(
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.ATTENTION_SUBSET_FEATURE_ORDER,
        )
        md("#### 4. Attention-Only Subset Summary By Held-Out Environment")
        display(ood_tables.style_metric_table(attention_env_table))
